In [3]:
#sample
xx = """Transat loss more than doubles as it works to complete Air Canada deal ; it works to complete Air Canada deal; Transat loss more than doubles; ; ; 34; 70; 0; 30; <e2>Transat loss more than doubles</e2> as <e1>it works to complete Air Canada deal</e1> 
""".split(";")
xx

['Transat loss more than doubles as it works to complete Air Canada deal ',
 ' it works to complete Air Canada deal',
 ' Transat loss more than doubles',
 ' ',
 ' ',
 ' 34',
 ' 70',
 ' 0',
 ' 30',
 ' <e2>Transat loss more than doubles</e2> as <e1>it works to complete Air Canada deal</e1> \n']

# Data Preprocessing

**Data processing approaches:**
1) providing sentence with cause and effect as seperate entity, 
``` json
{
  "instruction": "Identify the cause and effect in the following sentence. The cause is the event or situation that leads to another. The effect is the outcome or result.",
  "input": "Sentence: -----",
  "output": "Cause:---- . Effect: ----."
}
```
OR
```json
{
  "text": "Given the sentence: '----' Extract the cause and effect. Cause: --- Effect: ---"
}
```

2) Introduce the special tokens as <cause> and <effect>. For example:
```json
[
  {
    "instruction": "Identify the cause and effect in the following sentence by adding <cause> and <effect> tags around the respective parts. The cause is the event or situation that leads to another. The effect is the outcome or result.",
    "input": "Transat loss more than doubles as it works to complete Air Canada deal",
    "output": "<effect>Transat loss more than doubles</effect> as <cause>it works to complete Air Canada deal</cause>"
  },
  // ... more examples
]
```

This is a special dataset, involve causal inference fine-tuning. If the output is not consistent then we should try:
- Add adversarial/ contrastive example by Include No-Tag Examples strategy.
- balance the dataset after that. 
- Keep early stop to stop overfitting. 

In [1]:
SPECIAL_TOKENS = ["<cause>", "</cause>", "<effect>", "</effect>"]


In [8]:
from datasets import load_dataset, Dataset
import pandas as pd
def load_and_preprocess_csv(csv_path, max_samples=None):
    """
    Reads the CSV, preprocesses it into the instruction-input-output format,
    and returns a Hugging Face Dataset object.
    """
    print(f"Loading and preprocessing CSV from {csv_path}...")
    try:
        df = pd.read_csv(csv_path, sep=';', encoding='iso-8859-1')
        print(f"Successfully read {len(df)} rows from {csv_path}.")
    except FileNotFoundError:
        print(f"Error: The file {csv_path} was not found.")
        raise
    except Exception as e:
        print(f"Error reading CSV: {e}")
        raise

    if max_samples:
        print(f"Using a subset of {min(max_samples, len(df))} samples.")
        df = df.head(min(max_samples, len(df)))

    instruction_template = (
        "Identify the cause and effect in the following sentence by adding "
        "<cause></cause> and <effect></effect> tags around the respective parts. "
        "The cause is the event or situation that leads to another. "
        "The effect is the outcome or result."
    )

    processed_examples = []
    # replace <e1> with <cause> and <e2> with <effect>
    for index, row in df.iterrows():
        
        text = str(row.get(' Text',"")).strip()
        sentence = str(row.get(' Sentence', '')).strip()
        cause_text = str(row.get(' Cause', '')).strip()
        effect_text = str(row.get(' Effect', '')).strip()

        if not sentence or not cause_text or not effect_text:
            warnings_count += 1
            skipped_rows += 1
            continue

        input_for_llm = text # Original sentence is the "input" part of the prompt
        output_for_llm = sentence.replace("e1","cause").replace("e2","effect") # Start with original sentence to insert tags

        processed_examples.append({
            "instruction": instruction_template,
            "input": input_for_llm,      # Original sentence
            "output": output_for_llm     # Sentence with <cause>/<effect> tags
        })
    
    # Convert list of dicts to Hugging Face Dataset
    formatted_dataset = Dataset.from_list(processed_examples)
    print(f"Converted CSV to Hugging Face Dataset with {len(formatted_dataset)} examples.")
    return formatted_dataset

In [14]:
data_path = "/home/yash/Finetunning/FinCausal/data/train.csv"
data= load_and_preprocess_csv(data_path)
data[2]

Loading and preprocessing CSV from /home/yash/Finetunning/FinCausal/data/train.csv...
Successfully read 1750 rows from /home/yash/Finetunning/FinCausal/data/train.csv.
Converted CSV to Hugging Face Dataset with 1750 examples.


{'instruction': 'Identify the cause and effect in the following sentence by adding <cause></cause> and <effect></effect> tags around the respective parts. The cause is the event or situation that leads to another. The effect is the outcome or result.',
 'input': 'Florida is unique in that it also draws a large proportion of higher net-worth individuals  -  more than 85 percent of its net inflow of income came from people earning at least six-figures.',
 'output': '<cause>Florida is unique in that it also draws a large proportion of higher net-worth individuals</cause>  -  <effect>more than 85 percent of its net inflow of income came from people earning at least six-figures.</effect>'}

### Model respones:

``` 
GT
'<cause>Florida is unique in that it also draws a large proportion of higher net-worth individuals</cause>  -  <effect>more than 85 percent of its net inflow of income came from people earning at least six-figures.</effect>'


GPT-4o

Florida is unique in that it also draws a large proportion of higher net-worth individuals — <cause>more than 85 percent of its net inflow of income came from people earning at least six-figures</cause>, <effect>it draws a large proportion of higher net-worth individuals</effect>.
```

``` 
Llama2-7B, 13B

Florida is unique in that it also draws a large proportion of higher net-worth individuals, [cause] who earn at least six-figures, [effect] resulting in more than 85 percent of its net inflow of income coming from this demographic.
```

```
Mistral

<cause>Florida is unique in that it also draws a large proportion of higher net-worth individuals</cause> - <effect>more than 85 percent of its net inflow of income came from people earning at least six-figures</effect>.
```

In [15]:
data.train_test_split(test_size=0.1)

DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output'],
        num_rows: 1575
    })
    test: Dataset({
        features: ['instruction', 'input', 'output'],
        num_rows: 175
    })
})

In [17]:
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments, EarlyStoppingCallback, BitsAndBytesConfig
import torch
MODEL_NAME = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"

def load_model(model_id=MODEL_NAME, device= None, use_lora:bool=False):
    device = device #if device else "cuda" if torch.cuda.is_available() else "cpu"
    tokenizer = AutoTokenizer.from_pretrained(model_id,
                                            trust_remote_code=True,
                                            padding_side = "right")
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.add_eos_token
    
    special_tokens_list = ["<cause>", "</cause>", "<effect>", "</effect>"]

    num_added_toks = tokenizer.add_special_tokens({"additional_special_tokens": special_tokens_list})
    # print(f"Added {num_added_toks} special tokens: {special_tokens_list}")
    
    model_load_kwargs = {
    "trust_remote_code": True,
    "device_map": device,
    "torch_dtype":torch.float16 if torch.cuda.is_available() else None, # you can change to torch.bfloat16. My GPU does not supports it. 
    }
    model = AutoModelForCausalLM.from_pretrained(model_id, **model_load_kwargs)
    model.resize_token_embeddings(len(tokenizer)) # including new tokens
    
    if model.config.pad_token_id is None:
        model.config.pad_token_id = tokenizer.pad_token_id
    
    # # Apply LoRA if specified
    # if use_lora:
    #     lora_config = LoraConfig(
    #         r=8,
    #         lora_alpha=32,
    #         target_modules=["q_proj", "v_proj","k_proj"],
    #         lora_dropout=0.1,
    #         bias="none",
    #         task_type=TaskType.CAUSAL_LM
    #     )
    #     model = get_peft_model(model, lora_config)
    #     model.print_trainable_parameters()
    
                                                
    return model, tokenizer

model, tokenizer = load_model(MODEL_NAME,device = "cuda:3")


/home/yash/.pyenv/versions/3.11.9/envs/FT/lib/python3.11/site-packages/torch/cuda/__init__.py:734: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


In [19]:
tk = tokenizer(str(data[2]))
tk

{'input_ids': [151646, 13608, 54974, 1210, 364, 28301, 1437, 279, 5240, 323, 2456, 304, 279, 2701, 11652, 553, 7842, 220, 151665, 151666, 323, 220, 151667, 151668, 9492, 2163, 279, 19511, 5479, 13, 576, 5240, 374, 279, 1538, 476, 6534, 429, 11508, 311, 2441, 13, 576, 2456, 374, 279, 15274, 476, 1102, 15670, 364, 1355, 1210, 364, 57027, 374, 4911, 304, 429, 432, 1083, 26643, 264, 3460, 21117, 315, 5080, 4179, 2630, 2364, 7775, 220, 481, 220, 803, 1091, 220, 23, 20, 3266, 315, 1181, 4179, 4601, 363, 315, 7911, 3697, 504, 1251, 27644, 518, 3245, 4743, 12, 98027, 15670, 364, 3006, 1210, 364, 151665, 57027, 374, 4911, 304, 429, 432, 1083, 26643, 264, 3460, 21117, 315, 5080, 4179, 2630, 2364, 7775, 151666, 220, 481, 256, 151667, 6384, 1091, 220, 23, 20, 3266, 315, 1181, 4179, 4601, 363, 315, 7911, 3697, 504, 1251, 27644, 518, 3245, 4743, 12, 98027, 13, 151668, 8275], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,

In [20]:
tokenizer.decode(tk["input_ids"])

"<｜begin▁of▁sentence｜>{'instruction': 'Identify the cause and effect in the following sentence by adding <cause></cause> and <effect></effect> tags around the respective parts. The cause is the event or situation that leads to another. The effect is the outcome or result.', 'input': 'Florida is unique in that it also draws a large proportion of higher net-worth individuals  -  more than 85 percent of its net inflow of income came from people earning at least six-figures.', 'output': '<cause>Florida is unique in that it also draws a large proportion of higher net-worth individuals</cause>  -  <effect>more than 85 percent of its net inflow of income came from people earning at least six-figures.</effect>'}"